# Model comparison: BLIP vs Florence-2 vs Qwen2.5-VL

## Before you start (once)

1. **Get the latest code.** In the VS Code terminal: `git pull`
2. **Free the GPU.** If `app.py` is running, click its terminal and press **Ctrl+C**. Quit **Ollama** (taskbar arrow, right-click the llama, Quit). Check in the terminal with `nvidia-smi`: under 1500 MiB used.
3. **Pick the Python.** Top right of this notebook: **Select Kernel** > **Python Environments** > **.venv**. It should say `.venv (Python 3.12)`.
4. **Choose your photos.** Start the app (`python app.py`), write down the numbers of 5 to 10 photos ("Photo 2" = 2), then stop the app again with **Ctrl+C**. Pick a mix: mascot, meal, group, party, outdoors, blurry.

## Run it

5. Click **Restart** at the top (clears the old run).
6. In cell **2**, change `PHOTOS = [2, 9, 14, 23, 31]` to your photo numbers.
7. Click **Run All** at the top, or click each cell and press **Shift+Enter**. Cell 3 takes about a minute.

## Read the results

8. **Cell 4:** each photo with the 3 answers next to it. Look at the photo first, then judge each answer.
9. **Cells 5 and 6:** the charts (speed and length).
10. **Cell 7:** saves all answers to `outputs/model_answers.csv`.

## Score them in the sheet

11. Open **Spotlight AI - models** in Drive, tab **Model testing**.
12. For every photo and model fill in **Facts in photo** (before reading the answers), **Correct**, **Missing**, **Invented**, **Seconds**, the answer and your name.
13. Check the **Scoreboard** tab: it compares the models by itself.

If something fails, copy the last red line of the error and send it to Matty.

The photos are shown here on your laptop only. Git strips all outputs from notebooks when you commit (nbstripout), so the photos never reach GitHub.

In [ ]:
# 1. Load the code
import base64
import html
import io

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

import pipeline as p
from compare_models import run_blip, run_florence, run_qwen

MODELS = ["BLIP", "Florence-2", "Qwen2.5-VL"]
COLORS = {"BLIP": "#D9822B", "Florence-2": "#7B4FB8", "Qwen2.5-VL": "#2E8B57"}


def thumb(img, side=360):
    """Small JPEG of the photo for the page, kept in memory only."""
    small = img.copy()
    small.thumbnail((side, side))
    buf = io.BytesIO()
    small.save(buf, "JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()


def answer_html(model, text):
    """Qwen answers in fields (caption | activity | ...), shown as a small list."""
    if model == "Qwen2.5-VL":
        caption, *fields = text.split(" | ")
        items = "".join(f"<li>{html.escape(f)}</li>" for f in fields)
        return f"{html.escape(caption)}<ul style='margin:4px 0 0 16px'>{items}</ul>"
    return html.escape(text)


print("ready")

In [ ]:
# 2. Write the photo numbers you want to test (from the app, "Photo 2" = 2)
PHOTOS = [2, 9, 14, 23, 31]

all_files = sorted(f for f in p.DATA.iterdir() if f.suffix.lower() in p.EXTS)
files = [all_files[i] for i in PHOTOS]
images = [p.load_image(f) for f in files]
print(f"{len(images)} photos loaded")

In [ ]:
# 3. Run the 3 models (about 1 minute for 5 photos)
results = {
    "BLIP": run_blip(images),
    "Florence-2": run_florence(images),
    "Qwen2.5-VL": run_qwen(files),
}
print("done")

In [ ]:
# 4. Each photo next to what the 3 models said. Score these in the Model testing sheet
for k, photo in enumerate(PHOTOS):
    rows = "".join(
        f"<tr><td style='padding:6px 10px;vertical-align:top;border-left:6px solid {COLORS[m]}'>"
        f"<b>{m}</b><br><span style='color:#888'>{results[m][k][1]:.1f} s</span></td>"
        f"<td style='padding:6px 10px;vertical-align:top'>{answer_html(m, results[m][k][0])}</td></tr>"
        for m in MODELS)
    display(HTML(
        f"<div style='display:flex;gap:16px;align-items:flex-start;margin:18px 0'>"
        f"<div><h3 style='margin:0 0 6px'>Photo {photo}</h3>"
        f"<img src='data:image/jpeg;base64,{thumb(images[k])}' style='max-width:360px;border-radius:8px'></div>"
        f"<table style='border-collapse:collapse;max-width:760px'>{rows}</table></div>"))

In [ ]:
# 5. Charts: speed and length per model
table = pd.DataFrame([
    {"photo": photo, "model": m, "seconds": results[m][k][1], "words": len(results[m][k][0].split()),
     "answer": results[m][k][0]}
    for m in MODELS for k, photo in enumerate(PHOTOS)])
summary = table.groupby("model")[["seconds", "words"]].mean().reindex(MODELS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, title in [(axes[0], "seconds", "Seconds per photo (lower = faster)"),
                          (axes[1], "words", "Words per answer (more detail, more to check)")]:
    bars = ax.bar(summary.index, summary[column], color=[COLORS[m] for m in summary.index])
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(0, summary[column].max() * 1.2)
fig.suptitle(f"{len(PHOTOS)} photos: {', '.join(map(str, PHOTOS))}")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Seconds per photo, photo by photo (does one photo take much longer?)
pivot = table.pivot(index="photo", columns="model", values="seconds")[MODELS]
ax = pivot.plot.bar(figsize=(12, 4), color=[COLORS[m] for m in MODELS], rot=0)
ax.set_ylabel("seconds")
ax.set_xlabel("photo number")
ax.set_title("Seconds per photo for each model")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Save every answer to a file you can paste into the sheet
table.to_csv(p.OUT / "model_answers.csv", index=False)
print("Saved:", p.OUT / "model_answers.csv")